<a href="https://colab.research.google.com/github/AhmedCode110/AC-MOT/blob/main/notebooks/AC_MOT_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# AC-MOT portable Colab runner — v15
GitHub is the source of truth; Colab is only the runner.

v15 is portable across authorized Colab accounts. It automatically resolves an accessible VisDrone copy and a writable results folder in the current Google Drive account. Every stage explains what is running, why it is needed, elapsed time, and what remains. The v15 speed test reports live-frame processing FPS separately from JPEG dataset playback FPS.


## Stage 1/6 — Mount Google Drive
Mounts the current Google account's Drive. v15 can search MyDrive, Shared drives, and Drive shortcuts for the VisDrone dataset. Results are written to this account's Drive.


In [ ]:
import time
NOTEBOOK_START = time.perf_counter()
STAGE_TIMES = {}

def fmt_time(seconds):
    seconds = max(float(seconds), 0.0)
    if seconds < 60:
        return f"{seconds:.1f}s"
    m, s = divmod(int(round(seconds)), 60)
    if m < 60:
        return f"{m}m {s:02d}s"
    h, m = divmod(m, 60)
    return f"{h}h {m:02d}m"

def begin_stage(number, title, now, why, remaining):
    print("\n" + "=" * 96, flush=True)
    print(f"[{number}/6] {title}", flush=True)
    print(f"[NOW] {now}", flush=True)
    print(f"[WHY] {why}", flush=True)
    print(f"[REMAINING AFTER THIS] {remaining}", flush=True)
    return time.perf_counter()

def end_stage(number, started):
    elapsed = time.perf_counter() - started
    STAGE_TIMES[number] = elapsed
    print(
        f"[DONE {number}/6] elapsed={fmt_time(elapsed)} | "
        f"notebook_total={fmt_time(time.perf_counter()-NOTEBOOK_START)}",
        flush=True,
    )

t = begin_stage(
    1,
    "Mount Google Drive",
    "Connecting /content/drive to the Google account currently running this notebook.",
    "v15 needs an accessible VisDrone copy and a persistent results destination.",
    "GitHub auth -> dependencies -> portable config -> preflight -> AC-MOT run",
)
from google.colab import drive
drive.mount("/content/drive")
end_stage(1, t)


## Stage 2/6 — Authenticate and sync the private GitHub repo
The account must have a Colab Secret named `GITHUB_TOKEN`. The token must belong to a GitHub account that can read the private `AhmedCode110/AC-MOT` repository. The token is validated but never printed or stored in the clone URL/Git config.


In [ ]:
t = begin_stage(
    2,
    "Authenticate and sync GitHub",
    "Validating GITHUB_TOKEN, checking private-repo access, then cloning/pulling main.",
    "Any authorized GitHub account can run v15; an unauthorized account stops here with a clear error.",
    "dependencies -> portable config -> preflight -> AC-MOT run",
)

from pathlib import Path
import json, os, sys, subprocess, tempfile, urllib.request, urllib.error
from google.colab import userdata

REPO_URL = "https://github.com/AhmedCode110/AC-MOT.git"
REPO_API = "https://api.github.com/repos/AhmedCode110/AC-MOT"
USER_API = "https://api.github.com/user"
REPO = Path("/content/AC-MOT")

try:
    token = userdata.get("GITHUB_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "Colab Secret GITHUB_TOKEN is missing or Notebook access is disabled. "
        "Create the secret and enable Notebook access."
    ) from exc
if not token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN is empty.")

def github_json(url):
    request = urllib.request.Request(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "AC-MOT-Colab-v15",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            return response.status, json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        try:
            detail = json.loads(body).get("message", body)
        except Exception:
            detail = body
        return exc.code, {"message": detail}

sub = time.perf_counter()
print("[2.1/3] [NOW] Validating GITHUB_TOKEN...", flush=True)
user_status, user_data = github_json(USER_API)
if user_status != 200:
    raise RuntimeError(
        f"GITHUB_TOKEN authentication failed HTTP {user_status}: "
        f"{user_data.get('message')}"
    )
github_user = user_data.get("login")
print(
    f"[OK] token valid | GitHub user={github_user} | "
    f"elapsed={fmt_time(time.perf_counter()-sub)}",
    flush=True,
)

sub = time.perf_counter()
print("[2.2/3] [NOW] Verifying private AC-MOT repository access...", flush=True)
repo_status, repo_data = github_json(REPO_API)
if repo_status != 200:
    raise RuntimeError(
        f"GitHub account {github_user} cannot read AhmedCode110/AC-MOT "
        f"(HTTP {repo_status}: {repo_data.get('message')})."
    )
print(
    f"[OK] repo access confirmed | {repo_data.get('full_name')} | "
    f"elapsed={fmt_time(time.perf_counter()-sub)}",
    flush=True,
)

sub = time.perf_counter()
print("[2.3/3] [NOW] Syncing repository main branch...", flush=True)
with tempfile.TemporaryDirectory() as tmp:
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["ACMOT_GH_TOKEN"] = token
    env["ACMOT_GH_USER"] = github_user
    helper = Path(tmp) / "askpass"
    helper.write_text(
        '#!/usr/bin/env python3\n'
        'import os, sys\n'
        'prompt = sys.argv[1] if len(sys.argv) > 1 else ""\n'
        'print(os.environ["ACMOT_GH_USER"] if "Username" in prompt else os.environ["ACMOT_GH_TOKEN"])\n'
    )
    helper.chmod(0o700)
    env["GIT_ASKPASS"] = str(helper)
    command = ["git", "-c", "credential.helper="]
    if (REPO / ".git").is_dir():
        print("[GITHUB] Existing clone -> pull --ff-only origin main", flush=True)
        result = subprocess.run(
            command + ["-C", str(REPO), "pull", "--ff-only", "origin", "main"],
            env=env, text=True, capture_output=True,
        )
    else:
        print("[GITHUB] No clone -> clone main", flush=True)
        result = subprocess.run(
            command + ["clone", "--branch", "main", REPO_URL, str(REPO)],
            env=env, text=True, capture_output=True,
        )
    if result.stdout.strip():
        print(result.stdout, flush=True)
    if result.stderr.strip():
        print(result.stderr, flush=True)
    result.check_returncode()

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print(
    f"[OK] repository ready | commit={commit} | "
    f"sync_elapsed={fmt_time(time.perf_counter()-sub)}",
    flush=True,
)
end_stage(2, t)


## Stage 3/6 — Install the pinned environment
Installs `requirements.txt` and the pinned TrackEval revision. Network/package work has no trustworthy ETA, so the notebook shows real elapsed time instead of inventing one.


In [ ]:
t = begin_stage(
    3,
    "Install pinned dependencies and TrackEval",
    "Installing requirements.txt, then preparing the pinned TrackEval revision.",
    "Package drift can change FPS and tracking metrics.",
    "portable config -> preflight -> AC-MOT run",
)
sub = time.perf_counter()
print("[3.1/2] [NOW] pip install requirements | ETA=not reliably knowable", flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")],
    check=True,
)
print(
    f"[OK] requirements installed | elapsed={fmt_time(time.perf_counter()-sub)}",
    flush=True,
)

sub = time.perf_counter()
print("[3.2/2] [NOW] preparing pinned TrackEval | ETA=not reliably knowable", flush=True)
subprocess.run(
    [sys.executable, str(REPO / "scripts/setup_trackeval.py"), "/content/TrackEval"],
    check=True,
)
print(
    f"[OK] TrackEval ready | elapsed={fmt_time(time.perf_counter()-sub)}",
    flush=True,
)
end_stage(3, t)


## Stage 4/6 — Resolve the portable v15 configuration
Reads `configs/active_config.txt`. For v15, `dataset=AUTO` is resolved against the current account's accessible Drive. `output_root=AUTO` becomes a writable folder in the current account's MyDrive. The resolved paths are printed before any experiment starts.


In [ ]:
t = begin_stage(
    4,
    "Resolve active version and portable paths",
    "Loading active_config.txt, auto-finding VisDrone, and selecting this account's results root.",
    "Removes Ahmed-specific absolute Drive paths from v15.",
    "preflight -> AC-MOT run",
)
import uuid

CONFIG_NAME = (REPO / "configs" / "active_config.txt").read_text().strip()
CONFIG_FILE = REPO / "configs" / CONFIG_NAME
CFG = json.loads(CONFIG_FILE.read_text())

print(f"[CONFIG] file={CONFIG_NAME}", flush=True)
print(f"[CONFIG] version={CFG.get('version','legacy/unversioned')}", flush=True)
print(f"[CONFIG] mode={CFG['mode']}", flush=True)
print(
    f"[CONFIG] speedtest_script={CFG.get('speedtest_script','scripts/speedtest_top3.py')}",
    flush=True,
)
print(
    f"[CONFIG] target_fps={CFG.get('target_fps')} gate_frames={CFG.get('gate_frames')}",
    flush=True,
)

if CFG.get("portable"):
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))
    from portable_v15 import resolve_portable_config, portable_requirements_text
    print("[PORTABLE] " + portable_requirements_text(), flush=True)
    CFG = resolve_portable_config(CFG, verbose=True)

DATASET_ON_DRIVE = Path(CFG["dataset"])
RESULTS = Path(CFG["output_root"])
assert (DATASET_ON_DRIVE / "annotations").is_dir()
assert (DATASET_ON_DRIVE / "sequences").is_dir()
print(f"[OK] resolved dataset={DATASET_ON_DRIVE}", flush=True)
print(f"[OK] resolved results={RESULTS}", flush=True)

CONFIG_PATH = Path("/content") / ("acmot_config_" + uuid.uuid4().hex + ".json")
CONFIG_PATH.write_text(json.dumps(CFG, indent=2))
print(f"[OK] runtime config={CONFIG_PATH}", flush=True)
end_stage(4, t)


## Stage 5/6 — Preflight
Validates pinned packages, CUDA, Tesla T4, the resolved dataset, and the selected v15 runner before expensive work. If another account is missing access or hardware, it stops here with a specific error.


In [ ]:
t = begin_stage(
    5,
    "Environment and portability preflight",
    "Checking package pins, CUDA/T4, resolved paths, active version and runner.",
    "Failing here avoids wasting a long experiment run.",
    "AC-MOT run",
)
subprocess.run(
    [sys.executable, str(REPO / "scripts/run.py"), "--config", str(CONFIG_PATH), "--check"],
    check=True,
)
end_stage(5, t)


## Stage 6/6 — Run AC-MOT v15
v15 reports two different FPS values: `processing` is the decoded/live-frame AC-MOT pipeline and is used for the 25 FPS realtime gate; `dataset_playback_serial` additionally includes local JPEG read/decode and is reported separately. One fresh YOLO inference is still executed for every measured frame.


In [ ]:
t = begin_stage(
    6,
    "Run active AC-MOT version",
    "Starting scripts/run.py. v15 prints copy/decode/profile/FPS progress and ETA.",
    "This is the actual experiment; each run gets a new persistent Drive envelope.",
    "none — final stage",
)
print(
    f"[RUN] version={CFG.get('version')} | script={CFG.get('speedtest_script')} | "
    f"target={CFG.get('target_fps')} FPS | dataset={CFG.get('dataset')}",
    flush=True,
)
print(
    "[RUN] Realtime gate definition: decoded/live-frame processing. "
    "JPEG dataset decode is measured separately.",
    flush=True,
)
subprocess.run(
    [sys.executable, str(REPO / "scripts/run.py"), "--config", str(CONFIG_PATH)],
    check=True,
)
end_stage(6, t)

print("\n" + "=" * 96, flush=True)
print("[OK] ALL NOTEBOOK STAGES COMPLETED", flush=True)
print(f"[TOTAL] elapsed={fmt_time(time.perf_counter()-NOTEBOOK_START)}", flush=True)
print(f"[RESULTS ROOT] {CFG['output_root']}", flush=True)
print(
    "[STAGE TIMES] "
    + str({k: fmt_time(v) for k, v in STAGE_TIMES.items()}),
    flush=True,
)
